# Notebook 00: Setup check

**Goal (10 minutes, plus the linked tutorials if the tools are new to you):** make sure your environment works, learn where things live in this repo, and know where to learn the three tools every lesson assumes: the terminal, git and conda.

Run every cell with Shift+Enter. If any cell errors, stop and fix that before moving on; everything after depends on it.

In [1]:
# --- setup: make the repo importable and keep figures inline ---
import sys, warnings
from pathlib import Path
REPO = Path.cwd().parent if (Path.cwd() / "../rainier_sws").exists() else Path.cwd()
sys.path.insert(0, str(REPO))
warnings.filterwarnings("ignore")
%matplotlib inline
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
from rainier_sws import paths, data, measure, quality, stats
print("repo:", REPO)

repo: /home/claude/repos/rainier-sws-tutorials


## The tools (do these before or alongside week 1)

Everything in this repo assumes you can open a terminal, move around a filesystem, run a script, and commit your work with git. If any of that is new, the Denolle Lab keeps a list of the tutorials the group recommends, the [Group_Utils cookbook](https://github.com/Denolle-Lab/Group_Utils/blob/main/cookbook.md); the ones to do first, each about an afternoon, are:

- **The shell:** Software Carpentry, [The Unix Shell](https://swcarpentry.github.io/shell-novice/) (CC BY 4.0). Navigating, pipes, loops, and writing a shell script. You will write one in week 8 to run the benchmark on every station.
- **Version control:** Software Carpentry, [Version Control with Git](https://swcarpentry.github.io/git-novice/). Through "Collaborating" at least. House rule: commit at the end of every session, and keep your lab notebook (a markdown file) in this repo. You will need a [GitHub account](https://github.com/signup); make it with your UW email.
- **Python, if you want a refresher:** Software Carpentry, [Plotting and Programming in Python](http://swcarpentry.github.io/python-novice-gapminder/).
- **Conda environments:** [a short conda/mamba introduction](https://astrobiomike.github.io/unix/conda-intro). The one idea: an *environment* is a self-contained set of packages, so this repo's pinned versions cannot break anything else on your machine, and vice versa.
- **Jupyter:** [Get started with Jupyter notebooks](https://www.earthdatascience.org/courses/intro-to-earth-data-science/open-reproducible-science/jupyter-python/get-started-with-jupyter-notebook-for-python/) (Earth Lab, CC BY 4.0), if the notebook interface itself is new.

### Installing Miniconda and the environment

If `conda --version` does not work in a fresh terminal, install Miniconda (a minimal conda; the full Anaconda is not needed):

```bash
# macOS, Apple silicon
curl -O https://repo.anaconda.com/miniconda/Miniconda3-latest-MacOSX-arm64.sh
bash Miniconda3-latest-MacOSX-arm64.sh
# macOS, Intel:  Miniconda3-latest-MacOSX-x86_64.sh
# Linux:         Miniconda3-latest-Linux-x86_64.sh
# Windows:       download and run the .exe from https://docs.conda.io/en/latest/miniconda.html
```

Accept the licence, keep the default install location, answer yes to `conda init`, then **open a new terminal**. Then, once:

```bash
conda config --add channels conda-forge
git clone https://github.com/mhemmett/rainier-sws-tutorials.git
cd rainier-sws-tutorials
conda env create -f environment.yml      # about 5 minutes
conda activate rainier-sws
pytest -q
jupyter lab notebooks/
```

Every later session is just `conda activate rainier-sws` and `jupyter lab`. The cell below checks that you are in the right environment and that git can see this repo.

In [ ]:
import os, shutil, subprocess, sys
print("python     :", sys.executable)
print("conda env  :", os.environ.get("CONDA_DEFAULT_ENV", "(not a conda environment; did you run `conda activate rainier-sws`?)"))
print("git        :", shutil.which("git") or "not found: install git (https://swcarpentry.github.io/git-novice/)")
if shutil.which("git"):
    r = subprocess.run(["git", "-C", str(REPO), "log", "-1", "--format=%h %cs %s"], capture_output=True, text=True)
    print("last commit:", r.stdout.strip() or r.stderr.strip())

In [3]:
import obspy, numpy, pandas, matplotlib, scipy, sklearn, numba, swspy
for m in (obspy, numpy, pandas, matplotlib, scipy, sklearn, numba):
    print(f"{m.__name__:<12} {m.__version__}")
print(f"{'swspy':<12} {swspy.__version__}  (local copy, Hudson et al.)")
assert numpy.__version__.startswith("1."), "swspy needs numpy < 2 — create the env from environment.yml"
from rainier_sws import timing
print(f"{'chronos':<12} {'importable' if timing.CHRONOS_AVAILABLE else 'NOT installed: notebook 05b will use the local copies in rainier_sws/timing.py'}  (Kidiwela, UW)")

obspy        1.5.1
numpy        1.26.4
pandas       3.0.5
matplotlib   3.11.2
scipy        1.17.1
sklearn      1.9.1
numba        0.68.0
swspy        1.0.2  (local copy, Hudson et al.)
chronos      importable  (Kidiwela, UW)


## What is in this repository

```
rainier-sws-tutorials/
├── notebooks/        the lessons (you are here); numbered, do them in order
├── rainier_sws/      small helper package: data loading, the measurement, quality filters, statistics, timing
├── swspy/            the shear-wave splitting library (local copy of Hudson et al.'s SWSPy)
├── data/catalog/     earthquake catalogs and station coordinates (small CSVs)
├── data/waveforms/   cached seismograms (miniSEED), one folder per station
├── results/          Michael's preliminary measurements, one CSV per station
├── figures/          anything you save
└── tests/            small tests that prove the code does what the notebooks say
```

Rule of the house: **notebooks call functions in `rainier_sws/`; they do not define the pipeline themselves.** That way the code you read in a lesson is the code that produces the results, and a bug fixed once is fixed everywhere.

In [4]:
print("stations file :", paths.STATIONS_CSV.relative_to(REPO))
print("STAR catalog  :", paths.station_catalog_csv("STAR").relative_to(REPO))
print("STAR waveforms:", paths.station_waveform_dir("STAR").relative_to(REPO),
      "->", len(list(paths.station_waveform_dir("STAR").glob("*.mseed"))), "miniSEED files")
print("STAR results  :", paths.station_results_csv("STAR").relative_to(REPO))

stations file : data/catalog/rainier_catalog_stations.csv
STAR catalog  : data/catalog/rainier_catalog_strict_STAR.csv
STAR waveforms: data/waveforms/STAR -> 10 miniSEED files
STAR results  : results/sws_star_strict_results.csv


## Can this machine reach EarthScope?

The lessons work offline from the cached waveforms, but lesson 02 is more fun with a live connection. This cell just reports which case you are in.

In [5]:
try:
    from obspy.clients.fdsn import Client
    Client(data.FDSN_URL, timeout=20)
    print("EarthScope FDSN service reachable: live downloads will work.")
except Exception as exc:
    print("EarthScope not reachable from here (", type(exc).__name__, ") — notebooks will use the cache.")

EarthScope not reachable from here ( FDSNNoServiceException ) — notebooks will use the cache.


## Before you go on

- Skim the [ObsPy tutorial index](https://docs.obspy.org/tutorial/index.html). You will be sent to specific pages from the next notebooks.
- Open `rainier_sws/paths.py` and `rainier_sws/data.py` in an editor and read them. They are short.
- Read the section *What we are doing and why* in the README: it is the one-page version of the research project these lessons serve.
- Create your lab notebook: `notebooks/labnotes_<yourname>.md`, dated entries, commit it.

Next: `01_what_is_a_seismogram.ipynb`.